In [0]:

# IMPORTS E CONFIGURAÇÃO DOS BANCOS
from pyspark.sql.functions import col, hash, concat_ws, lit, expr, to_date, sequence, explode, year, month, dayofmonth, date_format, quarter, dayofweek, current_date
from pyspark.sql.types import StringType, IntegerType, DoubleType, DecimalType, LongType
from delta.tables import DeltaTable

# Define o catálogo workspace como padrão para garantir que todos os schemas sejam criados nele
spark.sql("USE CATALOG workspace")

banco_silver = "silver_olist"
banco_gold = "gold_olist"

# Garante a criação do banco da Gold no catálogo workspace
spark.sql(f"CREATE DATABASE IF NOT EXISTS {banco_gold}")

print("Iniciando a Construção da Camada Gold (Star Schema com SCD2 em Clientes) \n")

#DIMENSÃO CLIENTES (dim_customers) - IMPLEMENTAÇÃO SCD TIPO 2
print("Processando dim_customers (SCD2)...")
df_stg_customers = spark.read.table(f"{banco_silver}.stg_customers")

# Preparando os dados da Stage com as colunas de controle do SCD2
# Nota: Na SK combinamos o ID com a Cidade/Estado para garantir que se o cliente se mudar,
# a nova linha terá uma SK única e não colidirá com o histórico anterior.
df_stg_customers_prep = df_stg_customers.select(
    hash(concat_ws("||", col("customer_id"), col("customer_city"), col("customer_state"))).alias("sk_customer"),
    col("customer_id").alias("nk_customer"),
    col("customer_unique_id"),
    col("customer_zip_code_prefix"),
    col("customer_city"),
    col("customer_state"),
    current_date().alias("dt_inicio"),
    to_date(lit("2199-12-31")).alias("dt_fim"),
    lit(1).alias("fl_atual")
)

tabela_customers_gold = f"{banco_gold}.dim_customers"

# Se a tabela não existir, fazemos a carga inicial
if not spark.catalog.tableExists(tabela_customers_gold):
    print("[dim_customers] não encontrada. Criando tabela com estrutura SCD2...")
    df_stg_customers_prep.write.format("delta").mode("overwrite").saveAsTable(tabela_customers_gold)
else:
    print("[dim_customers] detetada. Executando o MERGE de SCD Tipo 2...")
    
    # 1. Carregamos a tabela atual da Gold para identificar o que mudou
    delta_customers = DeltaTable.forName(spark, tabela_customers_gold)
    
    # 2. Fazemos o MERGE para atualizar os registos antigos que mudaram de endereço
    # Se o ID do cliente bater, mas a cidade ou estado forem diferentes, desativamos o antigo
    regras_SCD2 = """
        target.nk_customer = source.nk_customer 
        AND target.fl_atual = 1 
        AND (target.customer_city != source.customer_city OR target.customer_state != source.customer_state)
    """
    
    # Atualiza as linhas que mudaram para expirar o histórico (fl_atual = 0)
    delta_customers.alias("target") \
        .merge(df_stg_customers_prep.alias("source"), "target.nk_customer = source.nk_customer") \
        .whenMatchedUpdate(
            condition="target.fl_atual = 1 AND (target.customer_city != source.customer_city OR target.customer_state != source.customer_state)",
            set={"dt_fim": current_date(), "fl_atual": lit(0)}
        ) \
        .execute()
        
    # 3. Inserimos as novas linhas (seja cliente novo ou o novo endereço do cliente antigo)
    # Filtramos apenas o que não existe ativo com aquela SK específica de endereço
    df_novos_ou_mudaram = df_stg_customers_prep.join(
        spark.read.table(tabela_customers_gold).filter("fl_atual = 1"),
        on="sk_customer",
        how="left_anti"
    )
    
    df_novos_ou_mudaram.write.format("delta").mode("append").saveAsTable(tabela_customers_gold)
    print("Histórico SCD2 atualizado com sucesso em dim_customers!")


# ==============================================================================
# 3. DIMENSÃO VENDEDORES (dim_sellers) - SCD1 (Overwrite)
# ==============================================================================
print("Construindo dim_sellers...")
df_stg_sellers = spark.read.table(f"{banco_silver}.stg_sellers")

df_dim_sellers = df_stg_sellers.select(
    hash(concat_ws("||", col("seller_id"))).alias("sk_seller"),
    col("seller_id").alias("nk_seller"),
    col("seller_zip_code_prefix"),
    col("seller_city"),
    col("seller_state")
)
df_dim_sellers.write.format("delta").mode("overwrite").saveAsTable(f"{banco_gold}.dim_sellers")


# ==============================================================================
# 4. DIMENSÃO PRODUTOS (dim_products) - SCD1 (Overwrite)
# ==============================================================================
print("Construindo dim_products (com tradução de categoria integrada)...")
df_stg_products = spark.read.table(f"{banco_silver}.stg_products")
df_stg_translation = spark.read.table(f"{banco_silver}.stg_product_category_name_translation")

df_dim_products = df_stg_products.join(
    df_stg_translation, 
    on="product_category_name", 
    how="left"
).select(
    hash(concat_ws("||", col("product_id"))).alias("sk_product"),
    col("product_id").alias("nk_product"),
    col("product_category_name"),
    col("product_category_name_english").alias("product_category_name_en"),
    col("product_name_lenght"),
    col("product_description_lenght"),
    col("product_photos_qty"),
    col("product_weight_g"),
    col("product_length_cm"),
    col("product_height_cm"),
    col("product_width_cm")
)
df_dim_products.write.format("delta").mode("overwrite").saveAsTable(f"{banco_gold}.dim_products")


# ==============================================================================
# 5. TABELA FATO PEDIDOS / VENDAS (ft_orders)
# ==============================================================================
print("Construindo a Tabela Fato ft_orders...")
df_stg_orders = spark.read.table(f"{banco_silver}.stg_orders")
df_stg_order_items = spark.read.table(f"{banco_silver}.stg_order_items")

# Join stg_customers to get customer location information
df_stg_customers = spark.read.table(f"{banco_silver}.stg_customers")

df_ft_orders = df_stg_order_items.join(
    df_stg_orders, 
    on="order_id", 
    how="inner"
).join(
    df_stg_customers,
    on="customer_id",
    how="inner"
).select(
    # SK única para a linha fática
    hash(concat_ws("||", col("order_id"), col("order_item_id"))).alias("sk_order_item"),
    
    # LIGAÇÃO COM SCD2: Como a fato liga com o cliente, precisamos garantir que ela aponte 
    # para a SK do endereço que ele tinha NO MOMENTO da compra.
    hash(concat_ws("||", col("customer_id"), col("customer_city"), col("customer_state"))).alias("sk_customer"),
    hash(concat_ws("||", col("product_id"))).alias("sk_product"),
    hash(concat_ws("||", col("seller_id"))).alias("sk_seller"),
    
    # SK numérica de Data para a dim_calendar
    hash(to_date(col("order_purchase_timestamp"))).alias("sk_date"),
    
    # Chaves Naturais para auditoria
    col("order_id").alias("nk_order"),
    col("order_item_id"),
    
    # Atributos e Timestamps originais
    col("order_status"),
    col("order_purchase_timestamp"),
    col("order_approved_at"),
    col("order_delivered_carrier_date"),
    col("order_delivered_customer_date"),
    col("order_estimated_delivery_date"),
    col("shipping_limit_date"),
    
    # Métricas prontas para as medidas DAX
    col("price").alias("vlr_price"),
    col("freight_value").alias("vlr_freight")
)
df_ft_orders.write.format("delta").mode("overwrite").saveAsTable(f"{banco_gold}.ft_orders")


# ==============================================================================
# CRIAÇÃO DINÂMICA DA DIMENSÃO CALENDÁRIO (dim_calendar)
# ==============================================================================
print("Construindo dim_calendar dinamicamente...")

anos_extremos = df_ft_orders.select(
    year(expr("MIN(order_purchase_timestamp)")).alias("min_year"),
    year(expr("MAX(order_purchase_timestamp)")).alias("max_year")
).collect()[0]

ano_inicio = f"{anos_extremos['min_year']}-01-01"
ano_fim = f"{anos_extremos['max_year']}-12-31"

df_calendar_base = spark.range(1).select(
    explode(sequence(to_date(lit(ano_inicio)), to_date(lit(ano_fim)), expr("interval 1 day"))).alias("dt_date")
)

df_dim_calendar = df_calendar_base.select(
    hash(col("dt_date")).alias("sk_date"), 
    col("dt_date"),
    year(col("dt_date")).alias("num_year"),
    month(col("dt_date")).alias("num_month"),
    dayofmonth(col("dt_date")).alias("num_day"),
    date_format(col("dt_date"), "MMMM").alias("txt_month_name"),
    quarter(col("dt_date")).alias("num_quarter"),
    dayofweek(col("dt_date")).alias("num_day_of_week"),
    expr("CASE WHEN month(dt_date) <= 6 THEN 1 ELSE 2 END").alias("num_semester")
)
df_dim_calendar.write.format("delta").mode("overwrite").saveAsTable(f"{banco_gold}.dim_calendar")


print("\n" + "="*60)
print("Camada Gold populada com Star Schema e SCD2 de Clientes!")
print("="*60)

Iniciando a Construção da Camada Gold (Star Schema com SCD2 em Clientes) 

Processando dim_customers (SCD2)...
[dim_customers] detetada. Executando o MERGE de SCD Tipo 2...
Histórico SCD2 atualizado com sucesso em dim_customers!
Construindo dim_sellers...
Construindo dim_products (com tradução de categoria integrada)...
Construindo a Tabela Fato ft_orders...
Construindo dim_calendar dinamicamente...

SUCESSO ABSOLUTO! Camada Gold populada com Star Schema e SCD2 de Clientes!


In [0]:
%sql
use catalog `workspace`; select * from `gold_olist`.`ft_orders` limit 100;

sk_order_item,sk_customer,sk_product,sk_seller,sk_date,nk_order,order_item_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,shipping_limit_date,vlr_price,vlr_freight
1393548000,251718884,-1169027519,-1878929958,97403799,0032d07457ae9c806c79368d7d9ce96b,1,delivered,2018-03-10T18:53:06.000Z,2018-03-10T19:48:19.000Z,2018-03-13T17:16:42.000Z,2018-04-29T21:08:59.000Z,2018-04-17T00:00:00.000Z,2018-03-15T19:28:51.000Z,159.0,27.19
-477557073,1520860788,-1211282008,1750800083,1714618410,0045e3085f083f0f38d24bb3f22e6593,1,delivered,2017-08-10T16:13:36.000Z,2017-08-11T07:15:16.000Z,2017-08-16T18:27:40.000Z,2017-08-29T20:03:08.000Z,2017-08-30T00:00:00.000Z,2017-08-17T07:15:16.000Z,116.9,13.84
-103605798,-638280550,-1748008892,-1050169889,134440160,0055b77cca3186676c147f532dd2547b,1,delivered,2018-02-19T21:47:44.000Z,2018-02-19T22:47:21.000Z,2018-02-20T19:35:24.000Z,2018-03-07T19:38:34.000Z,2018-03-14T00:00:00.000Z,2018-02-25T22:47:21.000Z,149.0,40.37
415865714,-1302024079,-512909391,1353220534,-1618576889,00995d799817ecc3bd2abd8fbe59c430,1,delivered,2018-07-26T11:25:18.000Z,2018-07-26T11:45:13.000Z,2018-08-01T14:05:00.000Z,2018-08-02T18:56:52.000Z,2018-08-08T00:00:00.000Z,2018-08-01T11:45:13.000Z,49.9,12.92
-1719171562,-595531930,483312312,1245319309,-893026955,00beb247698c6aae94e3f859d279e2cd,1,delivered,2018-05-24T22:21:17.000Z,2018-05-26T02:18:43.000Z,2018-05-28T15:04:00.000Z,2018-05-29T19:11:16.000Z,2018-06-11T00:00:00.000Z,2018-05-29T02:18:43.000Z,45.0,7.39
292774897,-341507202,-1081358536,144317337,-1616174864,00f8fc0cb9d9b7813f1f54fbd75426f1,1,delivered,2018-01-22T10:45:15.000Z,2018-01-22T14:16:52.000Z,2018-01-23T22:19:43.000Z,2018-02-02T01:02:38.000Z,2018-02-14T00:00:00.000Z,2018-01-26T14:16:49.000Z,149.0,14.77
-1980772613,305972266,1369999811,770132495,-104448189,0115f8bb1dc16d3fac863ce1deb037b6,1,delivered,2017-06-30T19:13:15.000Z,2017-06-30T19:25:18.000Z,2017-07-03T11:59:40.000Z,2017-07-07T17:12:36.000Z,2017-07-21T00:00:00.000Z,2017-07-07T19:25:18.000Z,119.99,16.47
-469857805,1295056048,-2084523476,-1776274857,1909555159,01c4766d43d50d892132f7bb04d97300,1,delivered,2018-03-22T23:11:17.000Z,2018-03-22T23:27:34.000Z,2018-03-26T20:31:27.000Z,2018-04-04T20:52:56.000Z,2018-04-19T00:00:00.000Z,2018-03-28T23:27:34.000Z,35.89,18.23
720885761,1267492088,631370809,-202875264,860578801,01cc71a8154340ae61126b5ee9b48fef,1,delivered,2018-06-06T21:06:18.000Z,2018-06-06T21:16:02.000Z,2018-06-08T12:09:00.000Z,2018-06-11T19:12:53.000Z,2018-07-03T00:00:00.000Z,2018-06-14T21:16:02.000Z,44.9,8.64
-819179664,-438246066,2120747172,-202875264,-543656517,0217ad862ae7391cc3964c07ce3d6ea1,1,delivered,2017-11-29T14:43:25.000Z,2017-11-29T14:58:38.000Z,2017-12-01T21:14:41.000Z,2017-12-26T21:32:13.000Z,2017-12-26T00:00:00.000Z,2017-12-05T14:58:38.000Z,99.9,17.95


In [0]:
current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
print(f"Catálogo padrão atual: {current_catalog}\n")

Catálogo padrão atual: workspace

